# Benchmark character space — full-coverage comparison (v2)

**Question.** Where does TSBench-Forge sit relative to
[GIFT-Eval](https://hf.co/datasets/Salesforce/GiftEval),
[TIME](https://hf.co/datasets/Real-TSF/TIME) and
[BOOM](https://hf.co/datasets/Datadog/BOOM), measured the way the
[BOOM paper](https://arxiv.org/abs/2505.14766) (§4.3) measures it?

**What changed vs v1.** (1) The feature set is exactly the paper's six statistics —
first-lag ACF, ARCH-LM, spectral entropy, KPSS, flat spots, skew — computed on
z-normalised series; ARCH-LM and KPSS via `statsmodels`. (2) **Every series in every
benchmark** is included (v1 sampled 300/benchmark): all GIFT-Eval Parquet series,
all TIME variates, all BOOM variates, and every panel-expanded series in our catalog.

**Outputs.** Fig-5-style distribution panels per feature, the PCA scatter, and
quantified spread/overlap. Downloads ~16.5 GB of benchmark data on first run
(cached under `HF_HOME` afterwards).

In [1]:
import os, sys, time, warnings
from pathlib import Path
import numpy as np

warnings.filterwarnings("ignore")
_cands = [Path.cwd(), Path.cwd() / "src", Path.cwd().parent / "src",
          Path(os.environ.get("TSBENCH_SRC", "/root/TSBench-Forge/src"))]
SRC = next(p for p in _cands if (p / "scraped_source.py").exists())
os.chdir(SRC); sys.path.insert(0, str(SRC))
MIN_LEN = 30            # tsfeatures-style minimum for the test statistics
ARCH_LAGS = 12
print("src:", SRC)

src: /root/TSBench-Forge/src


## 1. The six BOOM-paper statistics

Each computed on the z-normalised series, mirroring §4.3:

| feature | implementation | reading |
|---|---|---|
| `acf1` | lag-1 autocorrelation | low = noisy local oscillation |
| `arch_lm` | R² of the ARCH regression (squared series on 12 lags of itself), the `tsfeatures::arch_stat` convention | heteroskedasticity / time-varying volatility |
| `spec_entropy` | Shannon entropy of the normalised periodogram | high = irregular, aperiodic |
| `kpss` | KPSS level statistic (`statsmodels`, auto lags) | high = nonstationary |
| `flat_spots` | longest exactly-constant run ÷ length | high = sparse/stuck metrics |
| `skew` | sample skewness (pre-normalisation shape is preserved by z-scoring) | asymmetric, bursty tails |

In [2]:
from scipy import stats as sstats
from statsmodels.tsa.stattools import kpss as _kpss

def _acf1(z):
    d = float(np.dot(z, z))
    return float(np.dot(z[:-1], z[1:]) / d) if d > 0 else 0.0

def _arch_lm(z, lags=ARCH_LAGS):
    """tsfeatures arch_stat: R^2 of x^2 regressed on `lags` lags of x^2."""
    x2 = z ** 2
    m = min(lags, len(x2) // 4)
    if m < 1: return 0.0
    Y = x2[m:]
    X = np.column_stack([x2[m - i - 1: len(x2) - i - 1] for i in range(m)])
    X = np.column_stack([np.ones(len(Y)), X])
    try:
        beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
        resid = Y - X @ beta
        tss = float(((Y - Y.mean()) ** 2).sum())
        return float(1.0 - resid @ resid / tss) if tss > 0 else 0.0
    except np.linalg.LinAlgError:
        return 0.0

def _spec_entropy(z):
    y = z * np.hanning(len(z))
    psd = np.abs(np.fft.rfft(y))[1:] ** 2
    s = psd.sum()
    if s <= 0: return 1.0
    p = psd / s
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / np.log(len(psd)))

def _flat_spots(x):
    n = len(x)
    d = np.flatnonzero(np.diff(x) != 0)
    bounds = np.concatenate(([-1], d, [n - 1]))
    return float(np.diff(bounds).max() / n)

def features(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < MIN_LEN: return None
    sd = x.std()
    if sd == 0: return None          # constant series carries no signal for 5/6 stats
    z = (x - x.mean()) / sd
    try:
        k = float(_kpss(z, regression="c", nlags="auto")[0])
    except Exception:
        k = np.nan
    f = (_acf1(z), _arch_lm(z), _spec_entropy(z), k, _flat_spots(x), float(sstats.skew(x)))
    return f if all(np.isfinite(v) for v in f) else None

FEATS = ["acf1", "arch_lm", "spec_entropy", "kpss", "flat_spots", "skew"]
print(features(np.random.default_rng(0).normal(size=512)))

(-0.018616600794983305, 0.02094648485716777, 0.926476901071138, 0.10112082893100323, 0.001953125, -0.16804180542019137)


## 2. Every series from every benchmark

Iterated file-by-file (nothing held in memory beyond the feature rows). Sources:
local `snapshot_download` caches of the three benchmark repos, and our catalog's
latest parquet per source (panel-split, densest-numeric-column, chronologically
sorted — the loader's own conventions).

In [3]:
from huggingface_hub import snapshot_download
import pandas as pd, pyarrow.parquet as pq
import datasets as hfds
import yaml

paths = {rid: Path(snapshot_download(rid, repo_type="dataset", max_workers=8))
         for rid in ("Salesforce/GiftEvalParquet", "Datadog/BOOM", "Real-TSF/TIME")}
rows, labels = [], []

def add(bench, series):
    f = features(series)
    if f is not None:
        rows.append(f); labels.append(bench)

t0 = time.time()
# ---- ours: all panel-expanded series, full length
catalog = yaml.safe_load(open("sources/sources.yaml"))
for s in catalog:
    if s.get("disabled"): continue
    d = Path("sources/data") / s["id"]
    files = sorted(d.glob("*.parquet"))
    if not files: continue
    df = pd.read_parquet(files[-1])
    ts = pd.to_datetime(df.get("timestamp"), errors="coerce", utc=True, format="mixed") if "timestamp" in df else None
    if ts is not None and ts.notna().mean() > 0.9 and not ts.is_monotonic_increasing:
        df = df.iloc[np.argsort(ts.to_numpy(), kind="stable")]
    pc = [c for c in df.columns if c.startswith("_panel_")]
    groups = [g for _, g in df.groupby(pc[0])] if pc else [df]
    for g in groups:
        vals, best = None, -1
        for c in g.columns:
            if c == "timestamp" or c.startswith("_panel_"): continue
            v = pd.to_numeric(g[c], errors="coerce").to_numpy()
            fin = int(np.isfinite(v).sum())
            if fin > best: best, vals = fin, v
        if vals is not None: add("TSBench-Forge", vals)
print(f"ours: {labels.count('TSBench-Forge')} series ({time.time()-t0:.0f}s)")

ours: 289 series (9s)


In [4]:
t0 = time.time()
# ---- GIFT-Eval: every row of every config's train.parquet (streamed —
# the largest configs would not fit in memory as one table)
gift_files = sorted(paths["Salesforce/GiftEvalParquet"].rglob("train.parquet"))
for j, f in enumerate(gift_files):
    pf = pq.ParquetFile(f)
    for batch in pf.iter_batches(columns=["history_value"], batch_size=64):
        for v in batch.column(0).to_pylist():
            add("GIFT-Eval", np.asarray(v, dtype=float))
    if (j + 1) % 20 == 0:
        print(f"  GIFT file {j+1}/{len(gift_files)} ({time.time()-t0:.0f}s)", flush=True)
print(f"GIFT-Eval: {labels.count('GIFT-Eval')} series ({time.time()-t0:.0f}s)")

  GIFT file 20/97 (344s)


  GIFT file 40/97 (9327s)


  GIFT file 60/97 (9610s)


  GIFT file 80/97 (14221s)


GIFT-Eval: 354444 series (15250s)


In [5]:
t0 = time.time()
# ---- TIME: every variate of every dataset shard
for f in sorted(paths["Real-TSF/TIME"].rglob("*.arrow")):
    d = hfds.Dataset.from_file(str(f))
    for row in d:
        tgt = row["target"]
        for v in (tgt if isinstance(tgt[0], (list, np.ndarray)) else [tgt]):
            add("TIME", np.asarray(v, dtype=float))
print(f"TIME: {labels.count('TIME')} series ({time.time()-t0:.0f}s)")

TIME: 6633 series (95s)


In [6]:
t0 = time.time()
# ---- BOOM: every variate of every dataset shard (2,807 dirs)
boom_files = sorted(paths["Datadog/BOOM"].rglob("*.arrow"))
for i, f in enumerate(boom_files):
    try:
        d = hfds.Dataset.from_file(str(f))
    except Exception:
        continue
    for row in d:
        tgt = row["target"]
        for v in (tgt if isinstance(tgt[0], (list, np.ndarray)) else [tgt]):
            add("BOOM", np.asarray(v, dtype=float))
    if (i + 1) % 500 == 0:
        print(f"  BOOM shard {i+1}/{len(boom_files)} ({time.time()-t0:.0f}s)", flush=True)
print(f"BOOM: {labels.count('BOOM')} series ({time.time()-t0:.0f}s)")

  BOOM shard 500/2807 (612s)


  BOOM shard 1000/2807 (969s)


  BOOM shard 1500/2807 (1129s)


  BOOM shard 2000/2807 (1229s)


  BOOM shard 2500/2807 (1855s)


BOOM: 32712 series (2253s)


In [7]:
X = np.array(rows, dtype=float); labels = np.array(labels)
print("feature matrix:", X.shape)
print("per benchmark:", {b: int((labels == b).sum()) for b in dict.fromkeys(labels)})

feature matrix: (394078, 6)
per benchmark: {np.str_('TSBench-Forge'): 289, np.str_('GIFT-Eval'): 354444, np.str_('TIME'): 6633, np.str_('BOOM'): 32712}


## 3. Fig-5-style feature distributions

One panel per statistic; the BOOM paper's reading keys: broader/shifted BOOM
distributions = irregularity and nonstationarity of observability data. Where the
TSBench-Forge curve tracks BOOM's tails, our live operational feeds share that
character; where it tracks GIFT-Eval/TIME, we cover the classical regimes.

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

COL = {"GIFT-Eval": "#8888dd", "TIME": "#66bb88", "BOOM": "#dd8866", "TSBench-Forge": "#222222"}
CLIP = {"acf1": (-1, 1), "arch_lm": (0, 1), "spec_entropy": (0, 1),
        "kpss": (0, 3), "flat_spots": (0, 1), "skew": (-8, 8)}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (ax, feat) in enumerate(zip(axes.ravel(), FEATS)):
    lo, hi = CLIP[feat]
    bins = np.linspace(lo, hi, 60)
    for bench in ("GIFT-Eval", "TIME", "BOOM", "TSBench-Forge"):
        v = np.clip(X[labels == bench, i], lo, hi)
        ax.hist(v, bins=bins, density=True, histtype="step", lw=2 if bench == "TSBench-Forge" else 1.3,
                color=COL[bench], label=bench if i == 0 else None)
    ax.set_title(feat); ax.set_yticks([])
fig.legend(loc="upper center", ncol=4, frameon=False)
outdir = SRC.parent / "notebooks" / "figures"; outdir.mkdir(exist_ok=True)
fig.savefig(outdir / "benchmark_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved", outdir / "benchmark_feature_distributions.png")

saved /root/TSBench-Forge/notebooks/figures/benchmark_feature_distributions.png


## 4. PCA on the six statistics (all series)

In [9]:
from sklearn.decomposition import PCA
import pandas as pd

med = np.median(X, axis=0)
iqr = np.subtract(*np.percentile(X, [75, 25], axis=0)); iqr[iqr == 0] = 1.0
Xz = np.clip((X - med) / iqr, -6, 6)
pca = PCA(n_components=4, random_state=0)
P = pca.fit_transform(Xz)
print("explained variance:", np.round(pca.explained_variance_ratio_, 3),
      "| PC1+PC2 =", round(float(pca.explained_variance_ratio_[:2].sum()), 3))
print("\nloadings:\n", pd.DataFrame(pca.components_[:2].T, index=FEATS, columns=["PC1", "PC2"]).round(2))

fig, ax = plt.subplots(figsize=(9.5, 7.5))
order = ("GIFT-Eval", "BOOM", "TIME", "TSBench-Forge")
rng = np.random.default_rng(0)
for bench in order:
    m = np.flatnonzero(labels == bench)
    if len(m) > 20000: m = rng.choice(m, 20000, replace=False)   # plot cap only; stats use all
    ax.scatter(P[m, 0], P[m, 1], s=5, alpha=0.35 if bench != "TSBench-Forge" else 0.9,
               c=COL[bench], label=f"{bench} (n={int((labels==bench).sum()):,})",
               edgecolors="none")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
ax.set_title("BOOM-paper feature PCA — every series in each benchmark")
ax.legend(frameon=False, markerscale=3); ax.grid(alpha=0.25)
fig.savefig(outdir / "benchmark_pca_full.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved", outdir / "benchmark_pca_full.png")

explained variance: [0.441 0.278 0.206 0.061] | PC1+PC2 = 0.72
loadings:
                PC1   PC2
acf1         -0.13  0.03
arch_lm      -0.11  0.02
spec_entropy  0.15 -0.06
kpss         -0.43  0.87
flat_spots    0.72  0.47
skew          0.49  0.11


saved /root/TSBench-Forge/notebooks/figures/benchmark_pca_full.png


## 5. Spread, centroids, coverage

In [10]:
from scipy.spatial import cKDTree
ours = P[labels == "TSBench-Forge"]
print(f"{'benchmark':<16} {'n':>8} {'spread (tr cov PC1-4)':>22} {'centroid dist to ours':>22}")
for bench in dict.fromkeys(labels):
    m = labels == bench
    print(f"{bench:<16} {int(m.sum()):>8,} {float(np.trace(np.cov(P[m].T))):>22.2f} "
          f"{float(np.linalg.norm(P[m].mean(0) - ours.mean(0))):>22.2f}")
tree = cKDTree(ours[:, :2])
for bench in ("GIFT-Eval", "TIME", "BOOM"):
    d, _ = tree.query(P[labels == bench][:, :2])
    print(f"{bench}: median NN-distance to a TSBench-Forge point = {np.median(d):.3f} "
          f"| 95th pct = {np.percentile(d, 95):.2f}")

benchmark               n  spread (tr cov PC1-4)  centroid dist to ours
TSBench-Forge         289                   5.27                   0.00
GIFT-Eval         354,444                   7.86                   0.45
TIME                6,633                   5.86                   1.66
BOOM               32,712                  14.97                   1.40
GIFT-Eval: median NN-distance to a TSBench-Forge point = 0.091 | 95th pct = 0.75
TIME: median NN-distance to a TSBench-Forge point = 0.092 | 95th pct = 0.60
BOOM: median NN-distance to a TSBench-Forge point = 0.145 | 95th pct = 1.51


## 6. Reading the results

- **Per-feature panels (§3)** are the direct BOOM-paper comparison: check whether our
  distribution shares BOOM's heavy lower tail on `acf1`, its near-zero `arch_lm`
  peak, elevated `spec_entropy`/`kpss`, and the `flat_spots`/`skew` tails — and
  simultaneously covers the tighter GIFT-Eval/TIME modes.
- **PCA (§4)** now weights benchmarks by their true series counts; GIFT-Eval's tens of
  thousands of series no longer get equal visual weight with a 300-series sample.
  The scatter subsamples for rendering only — every statistic uses all series.
- **Spread/coverage (§5)**: trace-of-covariance = character diversity per benchmark;
  NN-distances = how densely we cover each incumbent's occupied region.
- **Caveats**: features are per-variate (cross-variate structure — BOOM's
  high-cardinality multivariate relationships — is out of scope here); KPSS is the
  level ('c') variant; our catalog contributes far fewer series (hundreds vs tens of
  thousands), so compare distributions and occupied regions, not point densities.